In [1]:
import pandas as pd
import gzip
import re
import tarfile
import io
from tqdm import tqdm

from pysradb.sraweb import SRAweb

/home/galvanized_heart/.local/lib/python3.10/site-packages/pysradb/utils.py:14: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


# Investigate GEO datasets
In this notebook, we will be looking at data from `Himes BE, Jiang X, Wagner P, Hu R, Wang Q, Klanderman B, Whitaker RM, Duan Q, Lasky-Su J, Nikolos C, Jester W, Johnson M, Panettieri R Jr, Tantisira KG, Weiss ST, Lu Q. “RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Muscle Cells.” PLoS One. 2014 Jun 13;9(6):e99625. PMID: 24926665. GEO: GSE52778.` and we'll be building a csv for the number of transcript counts for each experiment and metadata to map experiments to their conditions. The data (`airway_1.28.0.tar.gz`, `GSE52778_raw_counts_GRCh38.p13_NCBI.tsv.gz`, `GSE52778_series_matrix.txt.gz`, and `Human.GRCh38.p13.annot.tsv.gz`) can be downloaded from https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE52778 

Given datasets (`airway_rawcounts.csv` and `airway_metadata.csv`) were sourced from https://4va.github.io/biodatasci/r-rnaseq-airway.html 

In [2]:
series_matrix_file_gz = "GSE52778_series_matrix.txt.gz"
sample_count_to_inspect = 3 
samples_inspected = 0

print(f"--- Inspecting: {series_matrix_file_gz} ---")

with gzip.open(series_matrix_file_gz, 'rt') as f:
    for line in f:
        line = line.strip()
        if line.startswith("!Sample_geo_accession"):
            if samples_inspected >= sample_count_to_inspect:
                break
            samples_inspected += 1
            print("\n--- New Sample ---")
            print(line)
        elif samples_inspected > 0 and samples_inspected <= sample_count_to_inspect:
            if line.startswith("!Sample_title") or \
               line.startswith("!Sample_characteristics_ch1") or \
               line.startswith("!Sample_relation"): # SRA relations often hold SRR
                print(line)
        if line.startswith("!series_matrix_table_begin"): # Stop before the data table
            break

--- Inspecting: GSE52778_series_matrix.txt.gz ---

--- New Sample ---
!Sample_geo_accession	"GSM1275862"	"GSM1275863"	"GSM1275864"	"GSM1275865"	"GSM1275866"	"GSM1275867"	"GSM1275868"	"GSM1275869"	"GSM1275870"	"GSM1275871"	"GSM1275872"	"GSM1275873"	"GSM1275874"	"GSM1275875"	"GSM1275876"	"GSM1275877"
!Sample_characteristics_ch1	"treatment: Untreated"	"treatment: Dexamethasone"	"treatment: Albuterol"	"treatment: Albuterol_Dexamethasone"	"treatment: Untreated"	"treatment: Dexamethasone"	"treatment: Albuterol"	"treatment: Albuterol_Dexamethasone"	"treatment: Untreated"	"treatment: Dexamethasone"	"treatment: Albuterol"	"treatment: Albuterol_Dexamethasone"	"treatment: Untreated"	"treatment: Dexamethasone"	"treatment: Albuterol"	"treatment: Albuterol_Dexamethasone"
!Sample_characteristics_ch1	"tissue: human airway smooth muscle cells"	"tissue: human airway smooth muscle cells"	"tissue: human airway smooth muscle cells"	"tissue: human airway smooth muscle cells"	"tissue: human airway smooth mus

In [3]:
raw_counts_file_gz = "GSE52778_raw_counts_GRCh38.p13_NCBI.tsv.gz"
lines_to_inspect = 10

print(f"\n--- Inspecting: {raw_counts_file_gz} ---")

with gzip.open(raw_counts_file_gz, 'rt') as f:
    for i, line in enumerate(f):
        if i >= lines_to_inspect:
            break
        print(f"Line {i+1}: {line.strip()}")


--- Inspecting: GSE52778_raw_counts_GRCh38.p13_NCBI.tsv.gz ---
Line 1: GeneID	GSM1275862	GSM1275863	GSM1275864	GSM1275865	GSM1275866	GSM1275867	GSM1275868	GSM1275869	GSM1275870	GSM1275871	GSM1275872	GSM1275873	GSM1275874	GSM1275875	GSM1275876	GSM1275877
Line 2: 100287102	6	8	8	3	5	10	11	7	6	8	4	10	2	1	2	2
Line 3: 653635	357	292	351	285	469	428	523	442	367	415	404	354	371	381	367	341
Line 4: 102466751	12	11	12	10	15	15	19	15	9	11	18	9	10	9	12	6
Line 5: 107985730	1	1	2	1	3	2	2	3	1	1	1	1	2	1	1	1
Line 6: 100302278	0	0	1	0	0	0	1	1	0	0	1	0	0	0	0	1
Line 7: 645520	7	2	4	4	6	2	3	6	2	2	3	6	6	3	8	3
Line 8: 79501	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0	0
Line 9: 100996442	24	22	30	25	46	41	47	48	27	27	25	27	57	35	46	30
Line 10: 729737	50	38	46	40	118	124	137	122	43	56	60	43	78	70	79	68


In [4]:
annot = "Human.GRCh38.p13.annot.tsv.gz"
lines_to_inspect = 10

print(f"\n--- Inspecting: {annot} ---")

with gzip.open(annot, 'rt') as f:
    for i, line in enumerate(f):
        if i >= lines_to_inspect:
            break
        print(f"Line {i+1}: {line.strip()}")


--- Inspecting: Human.GRCh38.p13.annot.tsv.gz ---
Line 1: GeneID	Symbol	Description	Synonyms	GeneType	EnsemblGeneID	Status	ChrAcc	ChrStart	ChrStop	Orientation	Length	GOFunctionID	GOProcessID	GOComponentID	GOFunction	GOProcess	GOComponent
Line 2: 100287102	DDX11L1	DEAD/H-box helicase 11 like 1 (pseudogene)		pseudo	ENSG00000290825	active	NC_000001.11	11874	14409	positive	1652
Line 3: 653635	WASH7P	WASP family homolog 7, pseudogene	FAM39F|WASH5P	pseudo		active	NC_000001.11	14362	29370	negative	1769
Line 4: 102466751	MIR6859-1	microRNA 6859-1	hsa-mir-6859-1	ncRNA	ENSG00000278267	active	NC_000001.11	17369	17436	negative	68
Line 5: 107985730	MIR1302-2HG	MIR1302-2 host gene		ncRNA		active	NC_000001.11	29926	31295	positive	538
Line 6: 100302278	MIR1302-2	microRNA 1302-2	MIRN1302-2|hsa-mir-1302-2	ncRNA	ENSG00000284332	active	NC_000001.11	30366	30503	positive	138		GO:0035195			miRNA-mediated gene silencing
Line 7: 645520	FAM138A	family with sequence similarity 138 member A	F379|FAM138F	ncRNA	EN

In [5]:
airway = "airway_1.28.0.tar.gz"
lines_to_inspect = 10

print(f"\n--- Inspecting contents of: {airway} ---")

# Open the gzipped file in binary mode ('rb') for tarfile
with gzip.open(airway, 'rb') as gz_file:
    # Open the tar archive from the decompressed gzip stream
    # 'r:' means read with transparent compression (auto-detects based on stream)
    # but since we already used gzip.open, 'r' or 'r:*' is fine.
    # Using 'r:' or 'r|*' is more general if tarfile itself handles gz.
    with tarfile.open(fileobj=gz_file, mode='r:') as tar_archive:
        print("Files in the archive:")
        members = tar_archive.getmembers()
        if not members:
            print("Archive is empty.")

        for member in members:
            print(f"  - {member.name} {'(Directory)' if member.isdir() else ''}")

            # If you want to inspect a specific file or all regular files:
            if member.isfile(): # Check if it's a regular file
                print(f"\n--- Inspecting first {lines_to_inspect} lines of {member.name} within {airway} ---")
                # Extract the file as a file-like object (binary stream)
                extracted_file_binary = tar_archive.extractfile(member)
                if extracted_file_binary: # Make sure extraction was successful
                    # Wrap the binary stream in a TextIOWrapper to read as text
                    # Use an encoding you expect, or 'utf-8' with error handling
                    try:
                        with io.TextIOWrapper(extracted_file_binary, encoding='utf-8', errors='replace') as text_f:
                            for i, line in enumerate(text_f):
                                if i >= lines_to_inspect:
                                    break
                                print(f"  Line {i+1}: {line.strip()}")
                    except UnicodeDecodeError as ude:
                        print(f"    Could not decode {member.name} as UTF-8: {ude}")
                        print(f"    It might be a binary file or use a different encoding.")
                    except Exception as e_inner:
                        print(f"    Error reading {member.name}: {e_inner}")
                else:
                    print(f"    Could not extract {member.name} for inspection.")
            print("---") # Separator between files


--- Inspecting contents of: airway_1.28.0.tar.gz ---
Files in the archive:
  - airway/DESCRIPTION 

--- Inspecting first 10 lines of airway/DESCRIPTION within airway_1.28.0.tar.gz ---
  Line 1: Package: airway
  Line 2: Title: RangedSummarizedExperiment for RNA-Seq in airway smooth muscle
  Line 3: cells, by Himes et al PLoS One 2014
  Line 4: Version: 1.28.0
  Line 5: Author: Michael Love
  Line 6: Maintainer: Michael Love <michaelisaiahlove@gmail.com>
  Line 7: Description: This package provides a RangedSummarizedExperiment object
  Line 8: of read counts in genes for an RNA-Seq experiment on four human
  Line 9: airway smooth muscle cell lines treated with dexamethasone.
  Line 10: Details on the gene model and read counting procedure are
---
  - airway/NAMESPACE 

--- Inspecting first 10 lines of airway/NAMESPACE within airway_1.28.0.tar.gz ---
  Line 1: exportPattern(".")
---
  - airway/build (Directory)
---
  - airway/build/vignette.rds 

--- Inspecting first 10 lines of airway/

# Create full airway metadata

In [6]:
def parse_series_matrix_for_metadata(series_matrix_file_gz):
    data_lines = {}

    # Open file and parse data
    with gzip.open(series_matrix_file_gz, 'rt') as f:
        for line in f:
            line = line.strip()
            if line.startswith("!Sample_") or line.startswith("!Series_"):
                parts = line.split('\t')
                header = parts[0]
                values = [val.strip('"') for val in parts[1:]]
                if header not in data_lines:
                    data_lines[header] = []
                data_lines[header].append(values)
            elif line.startswith("!series_matrix_table_begin"):
                break

    num_samples = len(data_lines["!Sample_geo_accession"][0])
    all_samples_data = []

    # 
    for i in range(num_samples):
        sample_dict = {}
        sample_dict['geo_id'] = data_lines["!Sample_geo_accession"][0][i]

        treatments_raw = [item_list[i] for item_list in data_lines.get("!Sample_characteristics_ch1", []) if item_list[i].lower().startswith("treatment:")]
        cell_lines_raw = [item_list[i] for item_list in data_lines.get("!Sample_characteristics_ch1", []) if item_list[i].lower().startswith("cell line:")]
        
        if treatments_raw:
            sample_dict['treatment_protocol'] = treatments_raw[0].replace("treatment: ", "").strip()
        if cell_lines_raw:
            sample_dict['cell_line'] = cell_lines_raw[0].replace("cell line: ", "").strip()
        
        sra_relations = [item_list[i] for item_list in data_lines.get("!Sample_relation", []) if "SRA: https" in item_list[i]]
        if sra_relations:
            srx_match = re.search(r'SRX\d+', sra_relations[0])
            if srx_match:
                sample_dict['srx_id'] = srx_match.group(0)
        
        all_samples_data.append(sample_dict)
        
    return pd.DataFrame(all_samples_data)

In [7]:
# Extract metadata
series_matrix_file_gz = "GSE52778_series_matrix.txt.gz"
metadata_with_srx_df = parse_series_matrix_for_metadata(series_matrix_file_gz)
display(metadata_with_srx_df)

,geo_id,treatment_protocol,cell_line,srx_id
0,GSM1275862,Untreated,N61311,SRX384345
1,GSM1275863,Dexamethasone,N61311,SRX384346
2,GSM1275864,Albuterol,N61311,SRX384347
3,GSM1275865,Albuterol_Dexamethasone,N61311,SRX384348
4,GSM1275866,Untreated,N052611,SRX384349
5,GSM1275867,Dexamethasone,N052611,SRX384350
6,GSM1275868,Albuterol,N052611,SRX384351
7,GSM1275869,Albuterol_Dexamethasone,N052611,SRX384352
8,GSM1275870,Untreated,N080611,SRX384353
9,GSM1275871,Dexamethasone,N080611,SRX384354


Search for SRR ids are prefered to GEO ids since they provide direct access to raw sequencing data, whereas GEO ids only offer descriptive metadata about the experiments.

In [8]:
srr_ids_list = []
db = SRAweb()

for srx_id in tqdm(metadata_with_srx_df['srx_id'], desc="Fetching SRR ids"):
    if pd.isna(srx_id):
        srr_ids_list.append(None)
        continue
    # The sra_metadata call can raise exceptions, which the user wants to see if they occur
    df_srx = db.sra_metadata(srx_id, detailed=False) 
    if not df_srx.empty and 'run_accession' in df_srx.columns:
        srr_ids_list.append(df_srx['run_accession'].iloc[0])
    else:
        srr_ids_list.append(None) # Explicitly append None if no run_accession
        print(f"Warning: No run_accession found for SRX {srx_id} using pysradb.")
            
metadata_with_srx_df['srr_id'] = srr_ids_list
display(metadata_with_srx_df)

Fetching SRR ids: 100%|██████████| 16/16 [00:14<00:00,  1.07it/s]


,geo_id,treatment_protocol,cell_line,srx_id,srr_id
0,GSM1275862,Untreated,N61311,SRX384345,SRR1039508
1,GSM1275863,Dexamethasone,N61311,SRX384346,SRR1039509
2,GSM1275864,Albuterol,N61311,SRX384347,SRR1039510
3,GSM1275865,Albuterol_Dexamethasone,N61311,SRX384348,SRR1039511
4,GSM1275866,Untreated,N052611,SRX384349,SRR1039512
5,GSM1275867,Dexamethasone,N052611,SRX384350,SRR1039513
6,GSM1275868,Albuterol,N052611,SRX384351,SRR1039514
7,GSM1275869,Albuterol_Dexamethasone,N052611,SRX384352,SRR1039515
8,GSM1275870,Untreated,N080611,SRX384353,SRR1039516
9,GSM1275871,Dexamethasone,N080611,SRX384354,SRR1039517


Extract entrez gene ids from GEO annotations and convert as many as possible to ensembl gene ids since they have a more precise gene model representation.

In [9]:
# Build Entrez to Ensembl gene id map

annotation_file_gz = "Human.GRCh38.p13.annot.tsv.gz"
annotation_df = pd.read_csv(annotation_file_gz, sep='\t', low_memory=False)
print("Annotation DataFrame columns:", annotation_df.columns)

# Correct column names based on your inspection:
entrez_col_annot = 'GeneID'
ensembl_col_annot = 'EnsemblGeneID'

# Create the mapping dictionary
annotation_df[entrez_col_annot] = annotation_df[entrez_col_annot].astype(str)
annotation_df = annotation_df.dropna(subset=[entrez_col_annot, ensembl_col_annot])

# Filter for valid Ensembl Gene IDs
annotation_df = annotation_df[annotation_df[ensembl_col_annot].astype(str).str.startswith('ENSG')]

id_map_df = annotation_df[[entrez_col_annot, ensembl_col_annot]].drop_duplicates(subset=[entrez_col_annot], keep='first')
entrez_to_ensembl_map = pd.Series(id_map_df[ensembl_col_annot].values, index=id_map_df[entrez_col_annot]).to_dict()

print(f"Created Entrez to Ensembl map. Example items: {list(entrez_to_ensembl_map.items())[:5]}")

Annotation DataFrame columns: Index(['GeneID', 'Symbol', 'Description', 'Synonyms', 'GeneType',
       'EnsemblGeneID', 'Status', 'ChrAcc', 'ChrStart', 'ChrStop',
       'Orientation', 'Length', 'GOFunctionID', 'GOProcessID', 'GOComponentID',
       'GOFunction', 'GOProcess', 'GOComponent'],
      dtype='object')
Created Entrez to Ensembl map. Example items: [('100287102', 'ENSG00000290825'), ('102466751', 'ENSG00000278267'), ('100302278', 'ENSG00000284332'), ('645520', 'ENSG00000237613'), ('79501', 'ENSG00000186092')]


In [10]:
# Collect Entrez ids in dataset
raw_counts_file_gz = "GSE52778_raw_counts_GRCh38.p13_NCBI.tsv.gz"
raw_counts_df = pd.read_csv(raw_counts_file_gz, sep='\t')
raw_counts_df["GeneID"].head()

0    100287102
1       653635
2    102466751
3    107985730
4    100302278
Name: GeneID, dtype: int64

In [11]:
# Map Entrez to Ensembl gene id
raw_counts_df['GeneID'] = raw_counts_df['GeneID'].astype(str)
raw_counts_df['ensgene'] = raw_counts_df['GeneID'].map(entrez_to_ensembl_map)
raw_counts_df['ensgene'].head()

0    ENSG00000290825
1                NaN
2    ENSG00000278267
3                NaN
4    ENSG00000284332
Name: ensgene, dtype: object

In [12]:
# Drop unmappable ids
raw_counts_mapped_df = raw_counts_df.dropna(subset=['ensgene'])
raw_counts_df.head()

,GeneID,GSM1275862,GSM1275863,GSM1275864,GSM1275865,GSM1275866,GSM1275867,GSM1275868,GSM1275869,GSM1275870,GSM1275871,GSM1275872,GSM1275873,GSM1275874,GSM1275875,GSM1275876,GSM1275877,ensgene
0,100287102,6,8,8,3,5,10,11,7,6,8,4,10,2,1,2,2,ENSG00000290825
1,653635,357,292,351,285,469,428,523,442,367,415,404,354,371,381,367,341,NaN
2,102466751,12,11,12,10,15,15,19,15,9,11,18,9,10,9,12,6,ENSG00000278267
3,107985730,1,1,2,1,3,2,2,3,1,1,1,1,2,1,1,1,NaN
4,100302278,0,0,1,0,0,0,1,1,0,0,1,0,0,0,0,1,ENSG00000284332


Use metadata to map GSM to SRR ids in our raw counts dataset.

In [13]:
# Rename GSM columns to SRR IDs
gsm_to_srr = dict(zip(metadata_with_srx_df['geo_id'], metadata_with_srx_df['srr_id']))
gsm_cols = [col for col in raw_counts_mapped_df.columns if col.startswith('GSM')]
raw_counts_mapped_df.rename(columns=gsm_to_srr, inplace=True)
raw_counts_mapped_df.head()

/tmp/ipykernel_151207/1488723805.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  raw_counts_mapped_df.rename(columns=gsm_to_srr, inplace=True)


,GeneID,SRR1039508,SRR1039509,SRR1039510,SRR1039511,SRR1039512,SRR1039513,SRR1039514,SRR1039515,SRR1039516,SRR1039517,SRR1039518,SRR1039519,SRR1039520,SRR1039521,SRR1039522,SRR1039523,ensgene
0,100287102,6,8,8,3,5,10,11,7,6,8,4,10,2,1,2,2,ENSG00000290825
2,102466751,12,11,12,10,15,15,19,15,9,11,18,9,10,9,12,6,ENSG00000278267
4,100302278,0,0,1,0,0,0,1,1,0,0,1,0,0,0,0,1,ENSG00000284332
5,645520,7,2,4,4,6,2,3,6,2,2,3,6,6,3,8,3,ENSG00000237613
6,79501,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,ENSG00000186092


Expand Ensembl list of genes with empty gene list from expert crafted dataset by adding zeros to genes that aren't present in the data collected from GEO.

In [14]:
df_given = pd.read_csv("airway_rawcounts.csv")
df_given.head()

,ensgene,SRR1039508,SRR1039509,SRR1039512,SRR1039513,SRR1039516,SRR1039517,SRR1039520,SRR1039521
0,ENSG00000000003,679,448,873,408,1138,1047,770,572
1,ENSG00000000005,0,0,0,0,0,0,0,0
2,ENSG00000000419,467,515,621,365,587,799,417,508
3,ENSG00000000457,260,211,263,164,245,331,233,229
4,ENSG00000000460,60,55,40,35,78,63,76,60


In [15]:
# Add missing genes with zeros (takes time to run)
ensgene_list = df_given['ensgene'].tolist()
ensgene_set = set(ensgene_list)
existing_ensgenes = raw_counts_mapped_df['ensgene'].tolist()

srr_cols = [col for col in raw_counts_mapped_df.columns if col.startswith('SRR')]

for ensgene in ensgene_list:
    if (ensgene != 0) and (ensgene not in existing_ensgenes):
        new_row = {'ensgene': ensgene}
        for col in srr_cols:
            new_row[col] = 0
        new_row['GeneID'] = 0
        raw_counts_mapped_df = pd.concat([raw_counts_mapped_df, pd.DataFrame([new_row])], ignore_index=True)

In [16]:
# Clean up raw counts dataset
raw_counts_mapped_df_sorted = raw_counts_mapped_df.sort_values(by='ensgene')
raw_counts_mapped_df_sorted = raw_counts_mapped_df_sorted.drop(columns=['GeneID'])
ensgene_col = raw_counts_mapped_df_sorted.pop('ensgene')
raw_counts_mapped_df_sorted.insert(0, 'ensgene', ensgene_col)
raw_counts_mapped_df_sorted.head()

,ensgene,SRR1039508,SRR1039509,SRR1039510,SRR1039511,SRR1039512,SRR1039513,SRR1039514,SRR1039515,SRR1039516,SRR1039517,SRR1039518,SRR1039519,SRR1039520,SRR1039521,SRR1039522,SRR1039523
26462,ENSG00000000003,708,470,768,475,905,1088,1009,1093,1331,1093,1133,1646,1301,1018,1159,886
26461,ENSG00000000005,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0
24708,ENSG00000000419,464,517,458,527,620,939,885,560,663,799,727,721,701,868,550,689
1953,ENSG00000000457,313,250,279,276,325,482,399,349,353,407,352,344,466,485,314,344
1952,ENSG00000000460,115,108,145,119,115,179,179,140,172,156,131,150,195,192,157,160


In [17]:
raw_counts_mapped_df_sorted.to_csv('airway_rawcounts_more_exps.csv', index=False)
raw_counts_mapped_df_sorted.shape

(66188, 17)

In [18]:
# Clean up metadata
geo_id_col = metadata_with_srx_df.pop('geo_id')
metadata_with_srx_df['geo_id'] = geo_id_col

In [19]:
metadata_with_srx_df.to_csv('airway_metadata_more_exps.csv', index=False)
metadata_with_srx_df.shape

(16, 5)